FINAL

In [104]:
import tkinter as tk
from tkinter import filedialog, messagebox, simpledialog, ttk
from PIL import Image, ImageTk, ImageDraw
import numpy as np
import cv2

# إعدادات الألوان الاحترافية (Professional Dark Theme) [cite: 310, 316]
BG_COLOR = "#2c3e50"
SIDEBAR_COLOR = "#34495e"
ACCENT_COLOR = "#3498db"
TEXT_COLOR = "#ecf0f1"
BTN_COLOR = "#2980b9"
RED_BTN = "#e74c3c"

class SimplePhotoshop:
    def __init__(self, root):
        self.root = root
        self.root.title("KSIU - Digital Image Processing Project 2")
        self.root.geometry("1200x850")
        self.root.configure(bg=BG_COLOR)

        # المتغيرات الأساسية للمشروع [cite: 146]
        self.original_image = None  
        self.processed_numpy = None 
        self.display_image = None
        
        # متغيرات القص والتحجيم [cite: 165]
        self.start_x = self.start_y = 0
        self.rect = None
        self.scale_w = self.scale_h = 1.0 

        self.create_layout()

    def create_layout(self):
        # --- Sidebar (شريط الأدوات الجانبي) [cite: 50, 313] ---
        self.side_panel = tk.Frame(self.root, width=280, bg=SIDEBAR_COLOR, padx=15, pady=15)
        self.side_panel.pack(side=tk.LEFT, fill=tk.Y)
        self.side_panel.pack_propagate(False)

        tk.Label(self.side_panel, text="PHOTO EDITOR", font=('Segoe UI', 15, 'bold'), 
                 bg=SIDEBAR_COLOR, fg=ACCENT_COLOR).pack(pady=(0, 10))

        # قسم الملفات [cite: 324, 327]
        self.create_section_label("File Operations")
        self.create_button("Open Image", self.load_image)
        self.create_button("Save Result", self.save_image)

        # قسم أدوات التعديل [cite: 163, 347]
        self.create_section_label("Editing Tools")
        self.create_button("Crop Image", self.start_crop_mode)
        self.create_button("Add Text (Click)", self.apply_add_text)

        # قسم الخوارزميات اليدوية (Manual) [cite: 171, 332]
        self.create_section_label("Manual Algorithms")
        self.create_button("Grayscale", self.apply_grayscale_manual)
        self.create_button("Edges (Sobel)", self.apply_edge_manual)
        self.create_button("Sharpen Filter", self.apply_sharpen_manual)
        self.create_button("Blur Filter", self.apply_blur_manual)

        # تحكم السطوع اليدوي [cite: 221, 345]
        tk.Label(self.side_panel, text="Brightness Intensity", font=('Segoe UI', 8), bg=SIDEBAR_COLOR, fg=TEXT_COLOR).pack(pady=(5, 0))
        self.bright_slider = tk.Scale(self.side_panel, from_=-100, to=100, orient=tk.HORIZONTAL, 
                                     bg=SIDEBAR_COLOR, fg=TEXT_COLOR, highlightthickness=0, length=200)
        self.bright_slider.pack(fill=tk.X, pady=2)
        self.create_button("Apply Brightness", self.apply_brightness_manual)

        # قسم المؤثرات الحديثة [cite: 224, 370]
        self.create_section_label("Modern Effects")
        self.create_button("Pencil Sketch", self.apply_pencil_sketch)
        self.create_button("Sepia Tone", self.apply_sepia)

        # --- زر Reset في أسفل القائمة الجانبية تماماً --- [cite: 169]
        tk.Frame(self.side_panel, height=1, bg="#555").pack(fill=tk.X, pady=10)
        self.create_button("RESET IMAGE", self.reset_image, color=RED_BTN)

        # --- منطقة العرض (Main Workspace) [cite: 312, 326] ---
        self.main_frame = tk.Frame(self.root, bg=BG_COLOR)
        self.main_frame.pack(side=tk.RIGHT, expand=True, fill=tk.BOTH)

        self.canvas = tk.Canvas(self.main_frame, bg="#1a252f", highlightthickness=0, cursor="cross")
        self.canvas.pack(expand=True, fill=tk.BOTH, padx=20, pady=20)

    def create_section_label(self, text):
        tk.Label(self.side_panel, text=text.upper(), font=('Segoe UI', 8, 'bold'), 
                 bg=SIDEBAR_COLOR, fg="#bdc3c7").pack(anchor="w", pady=(10, 2))

    def create_button(self, text, command, color=BTN_COLOR):
        btn = tk.Button(self.side_panel, text=text, command=command, bg=color, fg="white", 
                        font=('Segoe UI', 9), bd=0, pady=5, cursor="hand2")
        btn.pack(fill=tk.X, pady=2)

    def load_image(self):
        file_path = filedialog.askopenfilename()
        if file_path:
            pil_img = Image.open(file_path).convert('RGB')
            self.original_image = np.array(pil_img)
            self.processed_numpy = self.original_image.copy()
            self.update_canvas()

    def update_canvas(self):
        if self.processed_numpy is not None:
            img_pil = Image.fromarray(self.processed_numpy.astype('uint8'))
            
            # تحديث ديناميكي للمركز والأبعاد لضمان ظهور الصورة [cite: 312, 326]
            self.root.update()
            canvas_w = self.canvas.winfo_width()
            canvas_h = self.canvas.winfo_height()
            if canvas_w <= 1: canvas_w, canvas_h = 800, 600

            w, h = img_pil.size
            ratio = min(canvas_w/w, canvas_h/h)
            new_w, new_h = int(w*ratio), int(h*ratio)
            self.scale_w, self.scale_h = w / new_w, h / new_h
            
            img_pil = img_pil.resize((new_w, new_h), Image.LANCZOS)
            self.display_image = ImageTk.PhotoImage(img_pil, master=self.root)
            
            self.canvas.delete("all")
            self.canvas.create_image(canvas_w/2, canvas_h/2, image=self.display_image, anchor=tk.CENTER)

    # --- أدوات التعديل اليدوية [cite: 163, 402] ---
    def start_crop_mode(self):
        messagebox.showinfo("Crop", "Select the area by dragging the mouse.")
        self.canvas.bind("<ButtonPress-1>", self.on_crop_press)
        self.canvas.bind("<B1-Motion>", self.on_crop_move)
        self.canvas.bind("<ButtonRelease-1>", self.on_crop_release)

    def on_crop_press(self, event):
        self.start_x, self.start_y = event.x, event.y
        if self.rect: self.canvas.delete(self.rect)
        self.rect = self.canvas.create_rectangle(self.start_x, self.start_y, event.x, event.y, outline='red', width=2)

    def on_crop_move(self, event):
        self.canvas.coords(self.rect, self.start_x, self.start_y, event.x, event.y)

    def on_crop_release(self, event):
        if self.processed_numpy is not None:
            canvas_w = self.canvas.winfo_width()
            canvas_h = self.canvas.winfo_height()
            img_w_disp = self.display_image.width()
            img_h_disp = self.display_image.height()
            
            off_x = (canvas_w - img_w_disp) / 2
            off_y = (canvas_h - img_h_disp) / 2

            end_x, end_y = event.x, event.y
            x1 = int((min(self.start_x, end_x) - off_x) * self.scale_w)
            y1 = int((min(self.start_y, end_y) - off_y) * self.scale_h)
            x2 = int((max(self.start_x, end_x) - off_x) * self.scale_w)
            y2 = int((max(self.start_y, end_y) - off_y) * self.scale_h)
            
            h, w, _ = self.processed_numpy.shape
            x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
            if x2 > x1 and y2 > y1:
                self.processed_numpy = self.processed_numpy[y1:y2, x1:x2]
                self.update_canvas()
        self.canvas.delete(self.rect)
        self.canvas.unbind("<ButtonPress-1>")

    def apply_add_text(self):
        if self.processed_numpy is not None:
            messagebox.showinfo("Text Tool", "Click on the image to place text.")
            self.canvas.bind("<Button-1>", self.place_text_on_click)

    def place_text_on_click(self, event):
        text = simpledialog.askstring("Input", "Enter text:")
        if text and self.processed_numpy is not None:
            canvas_w = self.canvas.winfo_width()
            canvas_h = self.canvas.winfo_height()
            off_x = (canvas_w - self.display_image.width()) / 2
            off_y = (canvas_h - self.display_image.height()) / 2
            
            img_x = int((event.x - off_x) * self.scale_w)
            img_y = int((event.y - off_y) * self.scale_h)
            
            h, w, _ = self.processed_numpy.shape
            if 0 <= img_x <= w and 0 <= img_y <= h:
                img_pil = Image.fromarray(self.processed_numpy.astype('uint8'))
                draw = ImageDraw.Draw(img_pil)
                draw.text((img_x, img_y), text, fill=(255, 255, 0)) 
                self.processed_numpy = np.array(img_pil)
                self.update_canvas()
        self.canvas.unbind("<Button-1>")

    # --- الخوارزميات اليدوية (Manual) [cite: 171, 332, 404] ---
    def apply_grayscale_manual(self):
        if self.processed_numpy is not None:
            R, G, B = self.processed_numpy[:,:,0], self.processed_numpy[:,:,1], self.processed_numpy[:,:,2]
            gray = (0.299*R + 0.587*G + 0.114*B).astype('uint8') # [cite: 176, 398]
            self.processed_numpy = np.stack([gray]*3, axis=-1)
            self.update_canvas()

    def apply_brightness_manual(self):
        if self.processed_numpy is not None:
            val = self.bright_slider.get()
            res = self.processed_numpy.astype(np.int16) + val # [cite: 222, 399]
            self.processed_numpy = np.clip(res, 0, 255).astype('uint8')
            self.update_canvas()

    def apply_histogram_equalization_manual(self):
        if self.processed_numpy is not None:
            img_gray = (0.299*self.processed_numpy[:,:,0] + 0.587*self.processed_numpy[:,:,1] + 0.114*self.processed_numpy[:,:,2]).astype('uint8')
            hist, _ = np.histogram(img_gray.flatten(), 256, [0,256]) # [cite: 190]
            cdf = hist.cumsum() # [cite: 191]
            cdf_norm = (cdf - cdf.min()) * 255 / (cdf.max() - cdf.min()) # [cite: 192, 400]
            self.processed_numpy = np.stack([cdf_norm[img_gray].astype('uint8')]*3, axis=-1)
            self.update_canvas()

    def apply_edge_manual(self):
        if self.processed_numpy is not None:
            gray = cv2.cvtColor(self.processed_numpy, cv2.COLOR_RGB2GRAY).astype(float)
            Kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]) # Sobel Kernel [cite: 197]
            edges = cv2.filter2D(gray, -1, Kx) # [cite: 200, 401]
            self.processed_numpy = np.stack([np.clip(np.abs(edges), 0, 255)]*3, axis=-1).astype('uint8')
            self.update_canvas()

    def apply_blur_manual(self):
        if self.processed_numpy is not None:
            kernel = np.ones((3, 3), np.float32) / 9 # [cite: 203, 205]
            self.processed_numpy = cv2.filter2D(self.processed_numpy, -1, kernel)
            self.update_canvas()

    def apply_sharpen_manual(self):
        if self.processed_numpy is not None:
            kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]) # [cite: 216]
            self.processed_numpy = cv2.filter2D(self.processed_numpy, -1, kernel) # [cite: 215]
            self.update_canvas()

    def apply_pencil_sketch(self):
        if self.processed_numpy is not None:
            gray, _ = cv2.pencilSketch(self.processed_numpy, sigma_s=60, sigma_r=0.07, shade_factor=0.05) # [cite: 246]
            self.processed_numpy = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
            self.update_canvas()

    def apply_sepia(self):
        if self.processed_numpy is not None:
            kernel = np.array([[0.393, 0.769, 0.189], [0.349, 0.686, 0.168], [0.272, 0.534, 0.131]]) # [cite: 233]
            self.processed_numpy = cv2.transform(self.processed_numpy, kernel)
            self.processed_numpy = np.clip(self.processed_numpy, 0, 255).astype('uint8')
            self.update_canvas()

    def reset_image(self):
        """إعادة ضبط الصورة للأصل [cite: 169]"""
        if self.original_image is not None:
            self.processed_numpy = self.original_image.copy()
            self.update_canvas()
            self.bright_slider.set(0)

    def save_image(self):
        if self.processed_numpy is not None:
            path = filedialog.asksaveasfilename(defaultextension=".png")
            if path: Image.fromarray(self.processed_numpy).save(path) # [cite: 327]

if __name__ == "__main__":
    root = tk.Tk()
    root.update_idletasks()
    app = SimplePhotoshop(root)
    root.mainloop()

In [9]:
import tkinter as tk
from tkinter import filedialog, messagebox, simpledialog
from PIL import Image, ImageTk, ImageDraw
import numpy as np
import cv2

# --- إعدادات الألوان ---
BG_COLOR = "#0f111a"        
SIDEBAR_COLOR = "#1a1d29"    
ACCENT_COLOR = "#2e3440"     # الأزرق الجليدي (Ice Blue)
TEXT_COLOR = "#d8dee9"       
BTN_COLOR = "#2e3440"        
RED_BTN = "#bf616a"          

class SimplePhotoshop:
    def __init__(self, root):
        self.root = root
        self.root.title("KSIU - Digital Image Processing Project 2")
        self.root.geometry("1200x850")
        self.root.configure(bg=BG_COLOR)

        self.original_image = None  
        self.processed_numpy = None 
        self.display_image = None
        
        self.start_x = self.start_y = 0
        self.rect = None
        self.scale_w = self.scale_h = 1.0 

        self.create_layout()

    def create_layout(self):
        # --- القائمة الجانبية (Sidebar) ---
        self.side_panel = tk.Frame(self.root, width=280, bg=SIDEBAR_COLOR, padx=12, pady=10)
        self.side_panel.pack(side=tk.LEFT, fill=tk.Y)
        self.side_panel.pack_propagate(False)

        # العنوان الرئيسي
        tk.Label(self.side_panel, text="PHOTO EDITOR", font=('Segoe UI', 17, 'bold'), 
                 bg=SIDEBAR_COLOR, fg=ACCENT_COLOR).pack(pady=(0, 10))

        # قسم العمليات
        self.create_section_label("File Operations")
        self.create_button("Open Image", self.load_image)
        self.create_button("Save Result", self.save_image)

        # قسم الأدوات
        self.create_section_label("Editing Tools")
        self.create_button("Crop Image", self.start_crop_mode)
        self.create_button("Add Text (Click)", self.apply_add_text)

        # قسم الخوارزميات
        self.create_section_label("Manual Algorithms")
        self.create_button("Grayscale", self.apply_grayscale_manual)
        self.create_button("Hist. Equalization", self.apply_histogram_equalization_manual)
        self.create_button("Edge Detection", self.apply_edge_manual)
        
        # فلاتر Sharpen و Blur
        filter_frame = tk.Frame(self.side_panel, bg=SIDEBAR_COLOR)
        filter_frame.pack(fill=tk.X, pady=2)
        tk.Button(filter_frame, text="Sharpen", command=self.apply_sharpen_manual, bg=BTN_COLOR, fg=TEXT_COLOR, font=('Segoe UI', 9, 'bold'), bd=0, pady=7, cursor="hand2").pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(0,2))
        tk.Button(filter_frame, text="Blur", command=self.apply_blur_manual, bg=BTN_COLOR, fg=TEXT_COLOR, font=('Segoe UI', 9, 'bold'), bd=0, pady=7, cursor="hand2").pack(side=tk.LEFT, fill=tk.X, expand=True)

        # تحكم السطوع
        tk.Label(self.side_panel, text="Brightness Intensity", font=('Segoe UI', 9), bg=SIDEBAR_COLOR, fg=TEXT_COLOR).pack(pady=(8, 0))
        self.bright_slider = tk.Scale(self.side_panel, from_=-100, to=100, orient=tk.HORIZONTAL, 
                                     bg=SIDEBAR_COLOR, fg=TEXT_COLOR, highlightthickness=0, 
                                     troughcolor=BG_COLOR, activebackground=ACCENT_COLOR)
        self.bright_slider.pack(fill=tk.X, pady=(0, 5))
        self.create_button("Apply Brightness", self.apply_brightness_manual)

        # قسم المؤثرات الحديثة
        self.create_section_label("Modern Effects")
        effects_frame = tk.Frame(self.side_panel, bg=SIDEBAR_COLOR)
        effects_frame.pack(fill=tk.X, pady=2)
        tk.Button(effects_frame, text="Sketch", command=self.apply_pencil_sketch, bg=BTN_COLOR, fg=TEXT_COLOR, font=('Segoe UI', 9, 'bold'), bd=0, pady=7, cursor="hand2").pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(0,2))
        tk.Button(effects_frame, text="Sepia", command=self.apply_sepia, bg=BTN_COLOR, fg=TEXT_COLOR, font=('Segoe UI', 9, 'bold'), bd=0, pady=7, cursor="hand2").pack(side=tk.LEFT, fill=tk.X, expand=True)

        # --- زر RESET (تم رفعه ليكون تحت قسم المؤثرات مباشرة) ---
        tk.Label(self.side_panel, text="", bg=SIDEBAR_COLOR).pack(pady=2) # فاصل صغير
        self.reset_btn = tk.Button(self.side_panel, text="RESET ALL CHANGES", command=self.reset_image, 
                                  bg=RED_BTN, fg="white", font=('Segoe UI', 10, 'bold'), 
                                  bd=0, pady=12, cursor="hand2")
        self.reset_btn.pack(fill=tk.X, pady=5)

        # --- منطقة العرض (تم تغيير الخلفية للأزرق الجليدي) ---
        self.main_frame = tk.Frame(self.root, bg=BG_COLOR)
        self.main_frame.pack(side=tk.RIGHT, expand=True, fill=tk.BOTH)

        # هنا تم تغيير لون الخلفية (bg) إلى ACCENT_COLOR
        self.canvas = tk.Canvas(self.main_frame, bg=ACCENT_COLOR, highlightthickness=0, cursor="cross")
        self.canvas.pack(expand=True, fill=tk.BOTH, padx=15, pady=15)

    def create_section_label(self, text):
        tk.Label(self.side_panel, text=text.upper(), font=('Segoe UI', 9, 'bold'), 
                 bg=SIDEBAR_COLOR, fg=ACCENT_COLOR).pack(anchor="w", pady=(10, 2))

    def create_button(self, text, command, color=BTN_COLOR):
        btn = tk.Button(self.side_panel, text=text, command=command, bg=color, fg=TEXT_COLOR, 
                        font=('Segoe UI', 10, 'bold'), bd=0, pady=8, cursor="hand2",
                        activebackground=ACCENT_COLOR)
        btn.pack(fill=tk.X, pady=3)

    # --- الدوال الأساسية (بدون تغيير في المنطق) ---
    def load_image(self):
        file_path = filedialog.askopenfilename()
        if file_path:
            pil_img = Image.open(file_path).convert('RGB')
            self.original_image = np.array(pil_img)
            self.processed_numpy = self.original_image.copy()
            self.update_canvas()

    def update_canvas(self):
        if self.processed_numpy is not None:
            img_pil = Image.fromarray(self.processed_numpy.astype('uint8'))
            self.root.update()
            canvas_w = self.canvas.winfo_width()
            canvas_h = self.canvas.winfo_height()
            if canvas_w <= 1: canvas_w, canvas_h = 800, 600
            w, h = img_pil.size
            ratio = min(canvas_w/w, canvas_h/h)
            new_w, new_h = int(w*ratio), int(h*ratio)
            self.scale_w, self.scale_h = w / new_w, h / new_h
            img_pil = img_pil.resize((new_w, new_h), Image.LANCZOS)
            self.display_image = ImageTk.PhotoImage(img_pil, master=self.root)
            self.canvas.delete("all")
            self.canvas.create_image(canvas_w/2, canvas_h/2, image=self.display_image, anchor=tk.CENTER)

    def start_crop_mode(self):
        messagebox.showinfo("Crop", "Select the area by dragging the mouse.")
        self.canvas.bind("<ButtonPress-1>", self.on_crop_press)
        self.canvas.bind("<B1-Motion>", self.on_crop_move)
        self.canvas.bind("<ButtonRelease-1>", self.on_crop_release)

    def on_crop_press(self, event):
        self.start_x, self.start_y = event.x, event.y
        if self.rect: self.canvas.delete(self.rect)
        self.rect = self.canvas.create_rectangle(self.start_x, self.start_y, event.x, event.y, outline="white", width=2)

    def on_crop_move(self, event):
        self.canvas.coords(self.rect, self.start_x, self.start_y, event.x, event.y)

    def on_crop_release(self, event):
        if self.processed_numpy is not None:
            canvas_w = self.canvas.winfo_width()
            canvas_h = self.canvas.winfo_height()
            off_x = (canvas_w - self.display_image.width()) / 2
            off_y = (canvas_h - self.display_image.height()) / 2
            x1 = int((min(self.start_x, event.x) - off_x) * self.scale_w)
            y1 = int((min(self.start_y, event.y) - off_y) * self.scale_h)
            x2 = int((max(self.start_x, event.x) - off_x) * self.scale_w)
            y2 = int((max(self.start_y, event.y) - off_y) * self.scale_h)
            h, w, _ = self.processed_numpy.shape
            x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
            if x2 > x1 and y2 > y1:
                self.processed_numpy = self.processed_numpy[y1:y2, x1:x2]
                self.update_canvas()
        self.canvas.delete(self.rect)
        self.canvas.unbind("<ButtonPress-1>")

    def apply_add_text(self):
        if self.processed_numpy is not None:
            messagebox.showinfo("Text Tool", "Click on the image to place text.")
            self.canvas.bind("<Button-1>", self.place_text_on_click)

    def place_text_on_click(self, event):
        text = simpledialog.askstring("Input", "Enter text:")
        if text and self.processed_numpy is not None:
            canvas_w = self.canvas.winfo_width()
            canvas_h = self.canvas.winfo_height()
            off_x = (canvas_w - self.display_image.width()) / 2
            off_y = (canvas_h - self.display_image.height()) / 2
            img_x = int((event.x - off_x) * self.scale_w)
            img_y = int((event.y - off_y) * self.scale_h)
            h, w, _ = self.processed_numpy.shape
            if 0 <= img_x <= w and 0 <= img_y <= h:
                img_pil = Image.fromarray(self.processed_numpy.astype('uint8'))
                draw = ImageDraw.Draw(img_pil)
                draw.text((img_x, img_y), text, fill=(255, 255, 0)) 
                self.processed_numpy = np.array(img_pil)
                self.update_canvas()
        self.canvas.unbind("<Button-1>")

    def apply_grayscale_manual(self):
        if self.processed_numpy is not None:
            R, G, B = self.processed_numpy[:,:,0], self.processed_numpy[:,:,1], self.processed_numpy[:,:,2]
            gray = (0.299*R + 0.587*G + 0.114*B).astype('uint8')
            self.processed_numpy = np.stack([gray]*3, axis=-1)
            self.update_canvas()

    def apply_brightness_manual(self):
        if self.processed_numpy is not None:
            val = self.bright_slider.get()
            res = self.processed_numpy.astype(np.int16) + val
            self.processed_numpy = np.clip(res, 0, 255).astype('uint8')
            self.update_canvas()

    def apply_histogram_equalization_manual(self):
        if self.processed_numpy is not None:
            img_gray = (0.299*self.processed_numpy[:,:,0] + 0.587*self.processed_numpy[:,:,1] + 0.114*self.processed_numpy[:,:,2]).astype('uint8')
            hist, _ = np.histogram(img_gray.flatten(), 256, [0,256])
            cdf = hist.cumsum()
            cdf_norm = (cdf - cdf.min()) * 255 / (cdf.max() - cdf.min())
            self.processed_numpy = np.stack([cdf_norm[img_gray].astype('uint8')]*3, axis=-1)
            self.update_canvas()

    def apply_edge_manual(self):
        if self.processed_numpy is not None:
            gray = cv2.cvtColor(self.processed_numpy, cv2.COLOR_RGB2GRAY).astype(float)
            Kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
            edges = cv2.filter2D(gray, -1, Kx)
            self.processed_numpy = np.stack([np.clip(np.abs(edges), 0, 255)]*3, axis=-1).astype('uint8')
            self.update_canvas()

    def apply_blur_manual(self):
        if self.processed_numpy is not None:
            kernel = np.ones((3, 3), np.float32) / 9 
            self.processed_numpy = cv2.filter2D(self.processed_numpy, -1, kernel)
            self.update_canvas()

    def apply_sharpen_manual(self):
        if self.processed_numpy is not None:
            kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
            self.processed_numpy = cv2.filter2D(self.processed_numpy, -1, kernel)
            self.update_canvas()

    def apply_pencil_sketch(self):
        if self.processed_numpy is not None:
            gray, _ = cv2.pencilSketch(self.processed_numpy, sigma_s=60, sigma_r=0.07, shade_factor=0.05)
            self.processed_numpy = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
            self.update_canvas()

    def apply_sepia(self):
        if self.processed_numpy is not None:
            kernel = np.array([[0.393, 0.769, 0.189], [0.349, 0.686, 0.168], [0.272, 0.534, 0.131]])
            self.processed_numpy = cv2.transform(self.processed_numpy, kernel)
            self.processed_numpy = np.clip(self.processed_numpy, 0, 255).astype('uint8')
            self.update_canvas()

    def reset_image(self):
        if self.original_image is not None:
            self.processed_numpy = self.original_image.copy()
            self.update_canvas()
            self.bright_slider.set(0)

    def save_image(self):
        if self.processed_numpy is not None:
            path = filedialog.asksaveasfilename(defaultextension=".png")
            if path: Image.fromarray(self.processed_numpy).save(path)

if __name__ == "__main__":
    root = tk.Tk()
    app = SimplePhotoshop(root)
    root.mainloop()

In [11]:
import tkinter as tk
from tkinter import filedialog, messagebox, simpledialog
from PIL import Image, ImageTk, ImageDraw
import numpy as np
import cv2

# --- إعدادات الألوان (Midnight Ice Theme) ---
BG_COLOR = "#0f111a"        
SIDEBAR_COLOR = "#1a1d29"    
ACCENT_COLOR = "#2e3440"     # الأزرق الجليدي لخلفية الصورة
TEXT_COLOR = "#d8dee9"       # أبيض ناصع
BTN_COLOR = "#2e3440"        
RED_BTN = "#bf616a"          

class SimplePhotoshop:
    def __init__(self, root):
        self.root = root
        self.root.title("KSIU - Digital Image Processing Project 2")
        self.root.geometry("1200x850")
        self.root.configure(bg=BG_COLOR)


        try:
            self.logo_img = tk.PhotoImage(file="ik.png") # استبدل your_logo.png باسم ملفك
            self.root.iconphoto(False, self.logo_img)
        except:
            print("Logo file not found")
        self.original_image = None  
        self.processed_numpy = None 
        self.display_image = None
        
        self.start_x = self.start_y = 0
        self.rect = None
        self.scale_w = self.scale_h = 1.0 

        self.create_layout()

    def create_layout(self):
        # --- القائمة الجانبية (Sidebar) ---
        self.side_panel = tk.Frame(self.root, width=280, bg=SIDEBAR_COLOR, padx=12, pady=10)
        self.side_panel.pack(side=tk.LEFT, fill=tk.Y)
        self.side_panel.pack_propagate(False)

        tk.Label(self.side_panel, text="PHOTO EDITOR", font=('Segoe UI', 17, 'bold'), 
                 bg=SIDEBAR_COLOR, fg="#ffffff").pack(pady=(0, 10))

        self.create_section_label("File Operations")
        self.create_button("Open Image", self.load_image)
        self.create_button("Save Result", self.save_image)

        self.create_section_label("Editing Tools")
        self.create_button("Crop Image", self.start_crop_mode)
        self.create_button("Add Text (Click)", self.apply_add_text)

        self.create_section_label("Manual Algorithms")
        self.create_button("Grayscale", self.apply_grayscale_manual)
        self.create_button("Hist. Equalization", self.apply_histogram_equalization_manual)
        self.create_button("Edge Detection", self.apply_edge_manual)
        
        filter_frame = tk.Frame(self.side_panel, bg=SIDEBAR_COLOR)
        filter_frame.pack(fill=tk.X, pady=2)
        tk.Button(filter_frame, text="Sharpen", command=self.apply_sharpen_manual, bg=BTN_COLOR, fg=TEXT_COLOR, font=('Segoe UI', 9, 'bold'), bd=0, pady=7, cursor="hand2").pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(0,2))
        tk.Button(filter_frame, text="Blur", command=self.apply_blur_manual, bg=BTN_COLOR, fg=TEXT_COLOR, font=('Segoe UI', 9, 'bold'), bd=0, pady=7, cursor="hand2").pack(side=tk.LEFT, fill=tk.X, expand=True)

        tk.Label(self.side_panel, text="Brightness Intensity", font=('Segoe UI', 9), bg=SIDEBAR_COLOR, fg="#ffffff").pack(pady=(8, 0))
        self.bright_slider = tk.Scale(self.side_panel, from_=-100, to=100, orient=tk.HORIZONTAL, 
                                     bg=SIDEBAR_COLOR, fg="#ffffff", highlightthickness=0, 
                                     troughcolor=BG_COLOR, activebackground=ACCENT_COLOR)
        self.bright_slider.pack(fill=tk.X, pady=(0, 5))
        self.create_button("Apply Brightness", self.apply_brightness_manual)

        self.create_section_label("Modern Effects")
        effects_frame = tk.Frame(self.side_panel, bg=SIDEBAR_COLOR)
        effects_frame.pack(fill=tk.X, pady=2)
        tk.Button(effects_frame, text="Sketch", command=self.apply_pencil_sketch, bg=BTN_COLOR, fg=TEXT_COLOR, font=('Segoe UI', 9, 'bold'), bd=0, pady=7, cursor="hand2").pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(0,2))
        tk.Button(effects_frame, text="Sepia", command=self.apply_sepia, bg=BTN_COLOR, fg=TEXT_COLOR, font=('Segoe UI', 9, 'bold'), bd=0, pady=7, cursor="hand2").pack(side=tk.LEFT, fill=tk.X, expand=True)

        tk.Label(self.side_panel, text="", bg=SIDEBAR_COLOR).pack(pady=2)
        self.reset_btn = tk.Button(self.side_panel, text="RESET ALL CHANGES", command=self.reset_image, 
                                  bg=RED_BTN, fg="#ffffff", font=('Segoe UI', 10, 'bold'), 
                                  bd=0, pady=12, cursor="hand2")
        self.reset_btn.pack(fill=tk.X, pady=5)

        self.main_frame = tk.Frame(self.root, bg=BG_COLOR)
        self.main_frame.pack(side=tk.RIGHT, expand=True, fill=tk.BOTH)

        self.canvas = tk.Canvas(self.main_frame, bg=ACCENT_COLOR, highlightthickness=0, cursor="cross")
        self.canvas.pack(expand=True, fill=tk.BOTH, padx=15, pady=15)

    def create_section_label(self, text):
        tk.Label(self.side_panel, text=text.upper(), font=('Segoe UI', 9, 'bold'), 
                 bg=SIDEBAR_COLOR, fg="#ffffff").pack(anchor="w", pady=(10, 2))

    def create_button(self, text, command, color=BTN_COLOR):
        btn = tk.Button(self.side_panel, text=text, command=command, bg=color, fg="#ffffff", 
                        font=('Segoe UI', 10, 'bold'), bd=0, pady=8, cursor="hand2",
                        activebackground=ACCENT_COLOR)
        btn.pack(fill=tk.X, pady=3)

    def load_image(self):
        file_path = filedialog.askopenfilename()
        if file_path:
            pil_img = Image.open(file_path).convert('RGB')
            self.original_image = np.array(pil_img)
            self.processed_numpy = self.original_image.copy()
            self.update_canvas()

    def update_canvas(self):
        if self.processed_numpy is not None:
            img_pil = Image.fromarray(self.processed_numpy.astype('uint8'))
            self.root.update()
            canvas_w = self.canvas.winfo_width()
            canvas_h = self.canvas.winfo_height()
            if canvas_w <= 1: canvas_w, canvas_h = 800, 600
            w, h = img_pil.size
            ratio = min(canvas_w/w, canvas_h/h)
            new_w, new_h = int(w*ratio), int(h*ratio)
            self.scale_w, self.scale_h = w / new_w, h / new_h
            img_pil = img_pil.resize((new_w, new_h), Image.LANCZOS)
            self.display_image = ImageTk.PhotoImage(img_pil, master=self.root)
            self.canvas.delete("all")
            self.canvas.create_image(canvas_w/2, canvas_h/2, image=self.display_image, anchor=tk.CENTER)

    def start_crop_mode(self):
        messagebox.showinfo("Crop", "Select the area by dragging the mouse.")
        self.canvas.bind("<ButtonPress-1>", self.on_crop_press)
        self.canvas.bind("<B1-Motion>", self.on_crop_move)
        self.canvas.bind("<ButtonRelease-1>", self.on_crop_release)

    def on_crop_press(self, event):
        self.start_x, self.start_y = event.x, event.y
        if self.rect: self.canvas.delete(self.rect)
        self.rect = self.canvas.create_rectangle(self.start_x, self.start_y, event.x, event.y, outline="white", width=2)

    def on_crop_move(self, event):
        self.canvas.coords(self.rect, self.start_x, self.start_y, event.x, event.y)

    def on_crop_release(self, event):
        if self.processed_numpy is not None:
            canvas_w = self.canvas.winfo_width()
            canvas_h = self.canvas.winfo_height()
            off_x = (canvas_w - self.display_image.width()) / 2
            off_y = (canvas_h - self.display_image.height()) / 2
            x1 = int((min(self.start_x, event.x) - off_x) * self.scale_w)
            y1 = int((min(self.start_y, event.y) - off_y) * self.scale_h)
            x2 = int((max(self.start_x, event.x) - off_x) * self.scale_w)
            y2 = int((max(self.start_y, event.y) - off_y) * self.scale_h)
            h, w, _ = self.processed_numpy.shape
            x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
            if x2 > x1 and y2 > y1:
                self.processed_numpy = self.processed_numpy[y1:y2, x1:x2]
                self.update_canvas()
        self.canvas.delete(self.rect)
        self.canvas.unbind("<ButtonPress-1>")

    def apply_add_text(self):
        if self.processed_numpy is not None:
            messagebox.showinfo("Text Tool", "Click on the image to place text.")
            self.canvas.bind("<Button-1>", self.place_text_on_click)

    def place_text_on_click(self, event):
        text = simpledialog.askstring("Input", "Enter text:")
        if text and self.processed_numpy is not None:
            canvas_w = self.canvas.winfo_width()
            canvas_h = self.canvas.winfo_height()
            off_x = (canvas_w - self.display_image.width()) / 2
            off_y = (canvas_h - self.display_image.height()) / 2
            img_x = int((event.x - off_x) * self.scale_w)
            img_y = int((event.y - off_y) * self.scale_h)
            h, w, _ = self.processed_numpy.shape
            if 0 <= img_x <= w and 0 <= img_y <= h:
                img_pil = Image.fromarray(self.processed_numpy.astype('uint8'))
                draw = ImageDraw.Draw(img_pil)
                draw.text((img_x, img_y), text, fill=(255, 255, 0)) 
                self.processed_numpy = np.array(img_pil)
                self.update_canvas()
        self.canvas.unbind("<Button-1>")

    def apply_grayscale_manual(self):
        if self.processed_numpy is not None:
            R, G, B = self.processed_numpy[:,:,0], self.processed_numpy[:,:,1], self.processed_numpy[:,:,2]
            gray = (0.299*R + 0.587*G + 0.114*B).astype('uint8')
            self.processed_numpy = np.stack([gray]*3, axis=-1)
            self.update_canvas()

    def apply_brightness_manual(self):
        if self.processed_numpy is not None:
            val = self.bright_slider.get()
            res = self.processed_numpy.astype(np.int16) + val
            self.processed_numpy = np.clip(res, 0, 255).astype('uint8')
            self.update_canvas()

    def apply_histogram_equalization_manual(self):
        if self.processed_numpy is not None:
            img_gray = (0.299*self.processed_numpy[:,:,0] + 0.587*self.processed_numpy[:,:,1] + 0.114*self.processed_numpy[:,:,2]).astype('uint8')
            hist, _ = np.histogram(img_gray.flatten(), 256, [0,256])
            cdf = hist.cumsum()
            cdf_norm = (cdf - cdf.min()) * 255 / (cdf.max() - cdf.min())
            self.processed_numpy = np.stack([cdf_norm[img_gray].astype('uint8')]*3, axis=-1)
            self.update_canvas()

    def apply_edge_manual(self):
        if self.processed_numpy is not None:
            gray = cv2.cvtColor(self.processed_numpy, cv2.COLOR_RGB2GRAY).astype(float)
            Kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
            edges = cv2.filter2D(gray, -1, Kx)
            self.processed_numpy = np.stack([np.clip(np.abs(edges), 0, 255)]*3, axis=-1).astype('uint8')
            self.update_canvas()

    def apply_blur_manual(self):
        if self.processed_numpy is not None:
            kernel = np.ones((3, 3), np.float32) / 9
            self.processed_numpy = cv2.filter2D(self.processed_numpy, -1, kernel)
            self.update_canvas()

    def apply_sharpen_manual(self):
        if self.processed_numpy is not None:
            kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
            self.processed_numpy = cv2.filter2D(self.processed_numpy, -1, kernel)
            self.update_canvas()

    def apply_pencil_sketch(self):
        if self.processed_numpy is not None:
            gray, _ = cv2.pencilSketch(self.processed_numpy, sigma_s=60, sigma_r=0.07, shade_factor=0.05)
            self.processed_numpy = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
            self.update_canvas()

    def apply_sepia(self):
        if self.processed_numpy is not None:
            kernel = np.array([[0.393, 0.769, 0.189], [0.349, 0.686, 0.168], [0.272, 0.534, 0.131]])
            self.processed_numpy = cv2.transform(self.processed_numpy, kernel)
            self.processed_numpy = np.clip(self.processed_numpy, 0, 255).astype('uint8')
            self.update_canvas()

    def reset_image(self):
        if self.original_image is not None:
            self.processed_numpy = self.original_image.copy()
            self.update_canvas()
            self.bright_slider.set(0)

    def save_image(self):
        if self.processed_numpy is not None:
            path = filedialog.asksaveasfilename(defaultextension=".png")
            if path: Image.fromarray(self.processed_numpy).save(path)

if __name__ == "__main__":
    root = tk.Tk()
    app = SimplePhotoshop(root)
    root.mainloop()